In [ ]:
#| default_exp latechunk_eval
#| eval: false

# Late Chunking Accuracy Eval

> Requires the eval extras: `uv sync --extra eval` (or `pip install litesearch[eval]`).

In [ ]:
#| eval: false
import os, certifi
os.environ.setdefault('SSL_CERT_FILE', certifi.where())
os.environ.setdefault('REQUESTS_CA_BUNDLE', certifi.where())
import requests as _r, urllib3 as _u
_u.disable_warnings()
_orig_send = _r.Session.send
_r.Session.send = lambda self, *a, **k: _orig_send(self, *a, **{**k, 'verify': False})

from litesearch import *
import numpy as np, json
from datasets import load_dataset
from fastcore.all import tuplify

/Users/71293/code/litesearch/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#| eval: false
_probe = load_dataset('dwzhu/LongEmbed', 'narrativeqa')
print(_probe)
for split in _probe: print(split, _probe[split].column_names, dict(list(_probe[split][0].items())[:3]).keys())

DatasetDict({
    corpus: Dataset({
        features: ['doc_id', 'text', 'qid'],
        num_rows: 355
    })
    queries: Dataset({
        features: ['doc_id', 'text', 'qid'],
        num_rows: 10449
    })
    qrels: Dataset({
        features: ['doc_id', 'text', 'qid'],
        num_rows: 10449
    })
})
corpus ['doc_id', 'text', 'qid'] dict_keys(['doc_id', 'text', 'qid'])
queries ['doc_id', 'text', 'qid'] dict_keys(['doc_id', 'text', 'qid'])
qrels ['doc_id', 'text', 'qid'] dict_keys(['doc_id', 'text', 'qid'])


In [ ]:
#| eval: false
TASKS = ['narrativeqa', '2wikimqa']

def load_longembed(task, max_docs=200, max_queries=100):
    'Load a LongEmbed task capped to max_docs/max_queries; keep only queries whose judged doc survives the cap.'
    ds = load_dataset('dwzhu/LongEmbed', task)
    corpus = {r['doc_id']: r['text'] for r in ds['corpus'].select(range(min(max_docs, len(ds['corpus']))))}
    queries = {r['qid']: r['text'] for r in ds['queries']}
    qrels = {}
    for r in ds['qrels']:
        qid, did = r['qid'], r['doc_id']
        if did in corpus and qid in queries: qrels.setdefault(qid, set()).add(did)
    queries = {q: queries[q] for q in list(qrels)[:max_queries]}
    qrels = {q: qrels[q] for q in queries}
    return corpus, queries, qrels

In [ ]:
#| eval: false
_corpus,_queries,_qrels = load_longembed('narrativeqa', max_docs=20, max_queries=5)
assert isinstance(_corpus, dict) and isinstance(_queries, dict) and isinstance(_qrels, dict)
assert len(_corpus) <= 20 and len(_queries) <= 5
for qid, docs in _qrels.items():
    assert qid in _queries
    assert all(d in _corpus for d in docs)
assert len(_qrels) >= 1

In [ ]:
#| eval: false
def agg_docs(hits):
    'Collapse ranked chunk hits to unique parent doc_ids, keeping best rank.'
    seen, out = set(), []
    for h in hits:
        d = json.loads(h['metadata'])['doc_id']
        if d not in seen: seen.add(d); out.append(d)
    return out

def recall_at_k(ranked, relevant, k):
    'Fraction of relevant docs present in the top-k ranking.'
    if not relevant: return 0.0
    return len(set(ranked[:k]) & relevant) / len(relevant)

def ndcg_at_k(ranked, relevant, k=10):
    'Binary-relevance nDCG@k.'
    dcg = sum(1.0/np.log2(i+2) for i,d in enumerate(ranked[:k]) if d in relevant)
    idcg = sum(1.0/np.log2(i+2) for i in range(min(len(relevant), k)))
    return float(dcg/idcg) if idcg else 0.0

In [ ]:
#| eval: false
_hits = [{'metadata': json.dumps({'doc_id':'A','chunk_idx':0})},
         {'metadata': json.dumps({'doc_id':'B','chunk_idx':3})},
         {'metadata': json.dumps({'doc_id':'A','chunk_idx':1})}]
assert agg_docs(_hits) == ['A','B']                     # dedup, keep first occurrence
assert recall_at_k(['A','B','C'], {'A','C'}, 3) == 1.0
assert recall_at_k(['A','B','C'], {'A','C'}, 1) == 0.5
assert abs(ndcg_at_k(['A','B'], {'A'}, 10) - 1.0) < 1e-9   # relevant doc first -> perfect
assert ndcg_at_k(['B','A'], {'A'}, 10) < 1.0               # relevant doc second -> discounted

In [ ]:
#| eval: false
# row builders (pure): produce litesearch rows for one method
def _rows_naive(corpus, enc):
    'Chunk-then-embed each chunk independently.'
    rows = []
    for did,txt in corpus.items():
        spans = chunk_spans(txt)
        embs = enc.encode_document([t for _,_,t in spans])
        for ci,((_,_,t),e) in enumerate(zip(spans,embs)):
            rows.append({'content':t,'embedding':e.tobytes(),'metadata':json.dumps({'doc_id':did,'chunk_idx':ci})})
    return rows

def _rows_fulldoc(corpus, enc):
    'One embedding per whole document.'
    rows = []
    for did,txt in corpus.items():
        e = enc.encode_document([txt])[0]
        rows.append({'content':txt,'embedding':e.tobytes(),'metadata':json.dumps({'doc_id':did,'chunk_idx':0})})
    return rows

def _rows_latechunk(corpus, lc):
    'Late chunking: embed whole doc, pool per chunk span.'
    rows = []
    for did,txt in corpus.items():
        spans = chunk_spans(txt)
        embs,_ = lc.encode_auto(txt, [(s,e) for s,e,_ in spans])
        for ci,((_,_,t),e) in enumerate(zip(spans,embs)):
            rows.append({'content':t,'embedding':e.tobytes(),'metadata':json.dumps({'doc_id':did,'chunk_idx':ci})})
    return rows

_SIT_CACHE = {}
def situate(chat, doc, chunk, doc_budget=2000):
    'Short doc-situating blurb via local Gemma; doc capped to fit the 4096-tok ctx, fresh conversation per call, non-fatal.'
    key = (hash(doc), hash(chunk))
    if key in _SIT_CACHE: return _SIT_CACHE[key]
    from rishi.core import resp_text
    chat.conv = chat._stack.enter_context(chat.engine.create_conversation())
    prompt = (f"<document>\n{doc[:doc_budget]}\n</document>\nHere is a chunk from it:\n<chunk>\n{chunk}\n</chunk>\n"
              "Give a short sentence situating this chunk within the document for search. Answer with only that sentence.")
    try: blurb = resp_text(chat(prompt)).strip()
    except Exception as e:
        import warnings; warnings.warn(f'situate failed ({type(e).__name__}); embedding chunk without context'); blurb = ''
    _SIT_CACHE[key] = blurb
    return blurb

def _rows_contextual(corpus, enc, chat, doc_budget=2000):
    'Contextual retrieval: prepend a Gemma situating blurb to each chunk before embedding.'
    rows = []
    for did,txt in corpus.items():
        spans = chunk_spans(txt)
        texts = [f"{situate(chat, txt, t, doc_budget)}\n{t}" for _,_,t in spans]
        embs = enc.encode_document(texts)
        for ci,(t,e) in enumerate(zip(texts,embs)):
            rows.append({'content':t,'embedding':e.tobytes(),'metadata':json.dumps({'doc_id':did,'chunk_idx':ci})})
    return rows

def _rows_for(method, corpus, enc, lc, chat, doc_budget=2000):
    'Dispatch to the row builder for a method name.'
    if method=='naive':      return _rows_naive(corpus, enc)
    if method=='fulldoc':    return _rows_fulldoc(corpus, enc)
    if method=='latechunk':  return _rows_latechunk(corpus, lc)
    if method=='contextual': return _rows_contextual(corpus, enc, chat, doc_budget)
    raise ValueError(f'unknown method {method!r}')

def _store(rows, db=None, name='store', ndim=768):
    'Insert rows into an ANN store (name) of db (in-memory if None) and build the HNSW index.'
    db = db if db is not None else database()
    st = db.get_store(name=name, ann=True, ndim=ndim, metric='cosine')
    st.insert_all(rows)
    st.rebuild_index()
    return db

# single-method in-memory builders (used by the unit tests below)
def build_naive(corpus, enc):            return _store(_rows_naive(corpus, enc), name='naive')
def build_fulldoc(corpus, enc):          return _store(_rows_fulldoc(corpus, enc), name='fulldoc')
def build_latechunk(corpus, lc):         return _store(_rows_latechunk(corpus, lc), name='latechunk')
def build_contextual(corpus, enc, chat): return _store(_rows_contextual(corpus, enc, chat), name='contextual')

In [ ]:
#| eval: false
_corpus = {'d1':'Cats are small mammals kept as pets. They purr when content.',
           'd2':'The Eiffel Tower is in Paris. It was built in 1889 for the World Fair.'}
_enc = FastEncode(model_dict=nomic_text_v15, max_seq_len=2048)
_lc  = AutoLateChunkFastEncode(model_dict=nomic_text_v15, max_seq_len=2048)
for _name,_db in [('naive',build_naive(_corpus,_enc)),('fulldoc',build_fulldoc(_corpus,_enc)),('latechunk',build_latechunk(_corpus,_lc))]:
    _q = 'where is the eiffel tower'
    _hits = _db.search(_q, _enc.encode_query(_q).tobytes(), columns=['metadata'], limit=10, table_name=_name)
    assert agg_docs(_hits)[0] == 'd2'          # obvious relevant doc ranks first

In [ ]:
#| eval: false
_senc= static_retrieval_embedder()
_senc.encode_document =_senc.encode_query = _senc.encode
benc = FastEncode(bge_model)

In [ ]:
#| eval: false
def evaluate(db, queries, qrels, enc, table_name='store', reranking=False, rerank_model=None):
    'Query-averaged nDCG@10 and Recall@{1,5,10} over one store (table_name).'
    ndcg=r1=r5=r10=0.0; n=len(queries)
    for qid,qtext in queries.items():
        hits = db.search(qtext, enc.encode_query(qtext).tobytes(), columns=['metadata'], limit=100,
             table_name=table_name, reranking=reranking, rerank_model=rerank_model)
        ranked = agg_docs(hits); rel = qrels[qid]
        ndcg += ndcg_at_k(ranked, rel, 10)
        r1 += recall_at_k(ranked, rel, 1); r5 += recall_at_k(ranked, rel, 5); r10 += recall_at_k(ranked, rel, 10)
    return {'ndcg@10':ndcg/n,'recall@1':r1/n,'recall@5':r5/n,'recall@10':r10/n}

DEFAULT_METHODS = ('naive','fulldoc','latechunk')
COLS = ('ndcg@10','recall@1','recall@5','recall@10')

def _clean_db_files(path):
    'Remove a task db and its usearch sidecars so a rebuild starts fresh.'
    import glob, os
    for f in [path] + glob.glob(f'{path}.*.usearch'):
        if os.path.exists(f): os.remove(f)

def build_eval_json(task, methods=DEFAULT_METHODS, queries=None, qrels=None, db_dir='.'):
    'Save queries/qrels/methods to eval.json for later evaluation.'
    json.dump({'methods':list(methods), 'queries':queries or {}, 'qrels':{q:sorted(v) for q,v in (qrels or {}).items()}},
              open(f'{db_dir}/eval_{task}.json','w'))

def build_task_db(task, max_docs=100, max_queries=50, methods=DEFAULT_METHODS, db_dir='.', doc_budget=2000, enc=None, lc=None):
    'Build one ANN store per method in eval_<task>.db; save queries/qrels/methods to eval_<task>.json. Returns db path.'
    corpus, queries, qrels = load_longembed(task, max_docs, max_queries)
    enc = enc or FastEncode(model_dict=nomic_text_v15, max_seq_len=8192)
    if 'latechunk' in methods and lc is None:
        assert getattr(enc,'model_dict',None) is not None, 'latechunk needs a FastEncode enc (with model_dict) or an explicit lc='
        lc = AutoLateChunkFastEncode(model_dict=enc.model_dict, max_seq_len=enc.max_seq_len)
    chat = None
    if 'contextual' in methods:
        from rishi.core import Chat; chat = Chat(cache_dir='.cache/litertlm')
    path = f'{db_dir}/eval_{task}.db'; _clean_db_files(path)
    db = database(path)
    for m in methods:
        rows = _rows_for(m, corpus, enc, lc, chat, doc_budget)
        _store(rows, db=db, name=m)
        print(f'{task}/{m}: {len(rows)} chunks', flush=True)
    json.dump({'methods':list(methods), 'queries':queries, 'qrels':{q:sorted(v) for q,v in qrels.items()}},
              open(f'{db_dir}/eval_{task}.json','w'))
    return path

def eval_task_db(task, db_dir='.', enc=None, reranking=False, rerank_model=None):
    'Score each method store in eval_<task>.db against saved queries/qrels. Returns {method: metrics}.'
    db = database(f'{db_dir}/eval_{task}.db')
    gt = json.load(open(f'{db_dir}/eval_{task}.json'))
    queries = gt['queries']; qrels = {q:set(v) for q,v in gt['qrels'].items()}
    enc = enc or FastEncode(model_dict=nomic_text_v15, max_seq_len=8192)
    return {m: evaluate(db, queries, qrels, enc, table_name=m, reranking=reranking,rerank_model=rerank_model) for m in gt['methods']}

def _print_table(agg):
    hdr = f"{'method':<12}" + "".join(f"{k:>10}" for k in COLS)
    print(hdr); print('-'*len(hdr))
    for m,r in agg.items(): print(f"{m:<12}" + "".join(f"{r[k]:>10.4f}" for k in COLS))
from fastcore.all import L
def run_comparison(tasks=TASKS, max_docs=100, max_queries=50, use_contextual=False, db_dir='.', rebuild=False, methods=DEFAULT_METHODS,
                   enc=None):
    'Build per-task dbs if missing, evaluate each, and print a method x metric table averaged over tasks.'
    import os
    methods = tuplify(methods) + (('contextual',) if use_contextual else ())
    per_task = {}
    tn=lambda t, r: f'{t}:{r}'
    for task in tasks:
        if rebuild or not os.path.exists(f'{db_dir}/eval_{task}.db'):
            build_task_db(task, max_docs, max_queries, methods=methods, db_dir=db_dir, enc=enc)
        per_task[tn(task,'base')] = eval_task_db(task, db_dir, enc)
        per_task[tn(task,'reranked')] = eval_task_db(task, db_dir, enc, reranking=True)
    agg = {}
    ts = [tn(t,i) for t in tasks for i in ['base','reranked']]
    for m in methods:
        if all(m in per_task[t] for t in ts):
            agg[tn(m,'base')] = {k: sum(per_task[tn(t,'base')][m][k] for t in tasks)/len(tasks) for k in COLS}
            agg[tn(m,'reranked')] = {k: sum(per_task[tn(t,'reranked')][m][k] for t in tasks)/len(tasks) for k in COLS}
        else: print(f'skipping {m!r}: not built in all task dbs (pass rebuild=True to include)')
    _print_table(agg)
    return agg

In [ ]:
#| eval: false
from fastcore.all import AttrDict
bge_small_instr = AttrDict(document="{text}",query="{text}")
bge_small_model = AttrDict(model='onnx-community/bge-small-en-v1.5-ONNX', onnx_path='onnx/model.onnx',prompt=bge_small_instr, tti=True)
bge_small = FastEncode(bge_small_model)
jina_sm_i = AttrDict(document="{text}",query="{text}")
jinasm_model = AttrDict(model='TonitoMC/jina-embeddings-v5-text-nano-classification-onnx', onnx_path='onnx/model.onnx',prompt=jina_sm_i)
jina_sm = FastEncode(jinasm_model)

In [ ]:
#| eval: false
run_comparison(db_dir='potion',methods=(DEFAULT_METHODS[0]),enc=_senc)
run_comparison(db_dir='bge',methods=(DEFAULT_METHODS[0]),enc=benc)

method         ndcg@10  recall@1  recall@5 recall@10
----------------------------------------------------
naive:base      0.1346    0.1200    0.1400    0.1600
naive:reranked    0.1606    0.1500    0.1700    0.1700
method         ndcg@10  recall@1  recall@5 recall@10
----------------------------------------------------
naive:base      0.6791    0.6400    0.6800    0.7400
naive:reranked    0.7423    0.6900    0.7600    0.7900


{'naive:base': {'ndcg@10': 0.6791472476991176,
  'recall@1': 0.64,
  'recall@5': 0.6799999999999999,
  'recall@10': 0.74},
 'naive:reranked': {'ndcg@10': 0.7423188053051863,
  'recall@1': 0.69,
  'recall@5': 0.76,
  'recall@10': 0.79}}

In [ ]:
#| eval: false
run_comparison(db_dir='jina',methods=(DEFAULT_METHODS[1]),enc=jina_sm)

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)' thrown while requesting HEAD https://huggingface.co/datasets/dwzhu/LongEmbed/resolve/10039a580487dacecf79db69166e17ace3ede392/LongEmbed.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since dwzhu/LongEmbed couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'narrativeqa' at /Users/71293/.cache/huggingface/datasets/dwzhu___long_embed/narrativeqa/0.0.0/10039a580487dacecf79db69166e17ace3ede392 (last modified on Mon Jul 20 12:39:55 2026).


KeyboardInterrupt: 

In [ ]:
#| eval: false
run_comparison(db_dir='bge_small',methods=(DEFAULT_METHODS[0]),enc=bge_small, reranking=True)

narrativeqa/naive: 9016 chunks


'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)' thrown while requesting HEAD https://huggingface.co/datasets/dwzhu/LongEmbed/resolve/10039a580487dacecf79db69166e17ace3ede392/LongEmbed.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since dwzhu/LongEmbed couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration '2wikimqa' at /Users/71293/.cache/huggingface/datasets/dwzhu___long_embed/2wikimqa/0.0.0/10039a580487dacecf79db69166e17ace3ede392 (last modified on Mon Jul 20 12:40:41 2026).


2wikimqa/naive: 980 chunks
method         ndcg@10  recall@1  recall@5 recall@10
----------------------------------------------------
naive           0.7234    0.6300    0.7700    0.8200


{'naive': {'ndcg@10': 0.7233786546280923,
  'recall@1': 0.63,
  'recall@5': 0.77,
  'recall@10': 0.8200000000000001}}

In [ ]:
#| eval: false
run_comparison(db_dir='bge_small',methods=(DEFAULT_METHODS[0]),enc=bge_small, reranking=True)

NameError: name 'bge_small' is not defined